In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from tqdm import tqdm
import time
from dataclasses import dataclass
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {device}")

@dataclass
class PowerSystem:
    n_buses: int
    n_gens: int
    n_lines: int
    bus_loads: np.ndarray
    gen_buses: np.ndarray
    gen_pmin: np.ndarray
    gen_pmax: np.ndarray
    gen_cost_c2: np.ndarray
    gen_cost_c1: np.ndarray
    gen_cost_c0: np.ndarray
    edge_index: np.ndarray
    ptdf_matrix: np.ndarray
    line_limits: np.ndarray
    b_matrix: np.ndarray
    admittance_matrix: np.ndarray

class DCOPFSolver:
    def __init__(self, power_system: PowerSystem):
        self.ps = power_system

    def solve(self, loads: np.ndarray):
        def objective(p_gen):
            return np.sum(self.ps.gen_cost_c2 * p_gen**2 + 
                         self.ps.gen_cost_c1 * p_gen + 
                         self.ps.gen_cost_c0)

        bounds = [(pmin, pmax) for pmin, pmax in zip(self.ps.gen_pmin, self.ps.gen_pmax)]
        constraints = [{'type': 'eq', 'fun': lambda p: p.sum() - loads.sum()}]

        total_load = loads.sum()
        capacities = self.ps.gen_pmax - self.ps.gen_pmin
        x0 = self.ps.gen_pmin + capacities * (total_load / capacities.sum())

        result = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)
        p_gen = result.x if result.success else x0
        return {'p_gen': p_gen, 'cost': objective(p_gen), 'success': result.success}

# ==============================================================================
# PROJECTION UTILITIES
# ==============================================================================

def soft_clamp(x, min_val, max_val, temperature=0.05):
    """Differentiable soft clamping"""
    x_normalized = (x - min_val) / (max_val - min_val + 1e-8)
    x_soft = torch.sigmoid((x_normalized - 0.5) / temperature) * 0.99 + 0.005
    return x_soft * (max_val - min_val) + min_val

def smart_project_to_feasible(p_gen, total_load, gen_pmin, gen_pmax, temperature=0.05):
    """Smart differentiable projection"""
    p_gen_clamped = soft_clamp(p_gen, gen_pmin, gen_pmax, temperature)
    
    current_total = p_gen_clamped.sum(dim=1, keepdim=True)
    mismatch = total_load - current_total
    
    capacity = gen_pmax - gen_pmin
    weights = capacity / capacity.sum()
    adjustment = mismatch * weights.unsqueeze(0)
    
    p_gen_adjusted = p_gen_clamped + adjustment
    p_gen_final = soft_clamp(p_gen_adjusted, gen_pmin, gen_pmax, temperature)
    
    return p_gen_final

def hard_project_to_feasible(p_gen, total_load, gen_pmin, gen_pmax):
    """Hard projection for inference"""
    p_gen = torch.clamp(p_gen, min=gen_pmin, max=gen_pmax)
    
    for _ in range(15):
        current_total = p_gen.sum(dim=1, keepdim=True)
        mismatch = total_load - current_total
        
        if mismatch.abs().max() < 0.005:
            break
        
        capacity = gen_pmax - gen_pmin
        weights = capacity / capacity.sum()
        adjustment = mismatch * weights.unsqueeze(0)
        p_gen = p_gen + adjustment
        p_gen = torch.clamp(p_gen, min=gen_pmin, max=gen_pmax)
    
    return p_gen

# ==============================================================================
# OPTIMIZED CFM REFINER
# ==============================================================================

class OptimizedCFMRefiner(nn.Module):
    """Enhanced CFM with residual connections"""
    
    def __init__(self, n_gens: int, gen_pmin, gen_pmax, hidden_dim: int = 256):
        super().__init__()
        
        self.n_gens = n_gens
        self.register_buffer('gen_pmin', torch.tensor(gen_pmin, dtype=torch.float32))
        self.register_buffer('gen_pmax', torch.tensor(gen_pmax, dtype=torch.float32))
        self.register_buffer('gen_capacity', torch.tensor(gen_pmax - gen_pmin, dtype=torch.float32))
        
        self.time_embed_dim = 64
        input_dim = n_gens + self.time_embed_dim + 1
        
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        
        self.res_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.SiLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim)
            ) for _ in range(3)
        ])
        
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, n_gens)
        )
        
    def time_embedding(self, t):
        """Sinusoidal time embedding"""
        half_dim = self.time_embed_dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb
    
    def normalize_dispatch(self, p_gen):
        return (p_gen - self.gen_pmin) / (self.gen_capacity + 1e-8)
    
    def denormalize_dispatch(self, p_gen_norm):
        return p_gen_norm * self.gen_capacity + self.gen_pmin
    
    def forward(self, x_t, t, total_load):
        """Predict vector field"""
        x_t_norm = self.normalize_dispatch(x_t)
        t_embed = self.time_embedding(t)
        total_capacity = self.gen_capacity.sum()
        load_norm = (total_load / total_capacity)
        
        features = torch.cat([x_t_norm, t_embed, load_norm], dim=-1)
        
        h = F.silu(self.input_layer(features))
        
        for res_block in self.res_blocks:
            h = h + res_block(h)
        
        v_t_norm = self.output_layer(h)
        v_t = v_t_norm * self.gen_capacity.unsqueeze(0) * 0.3
        
        return v_t
    
    def compute_flow_matching_loss(self, x_0, x_1, total_load):
        """Stabilized flow matching loss"""
        batch_size = x_0.shape[0]
        t = torch.rand(batch_size, device=x_0.device) * 0.8 + 0.1
        
        t_expanded = t[:, None]
        x_t = (1 - t_expanded) * x_0 + t_expanded * x_1
        u_t = x_1 - x_0
        v_t = self.forward(x_t, t, total_load)
        
        loss_fm = F.smooth_l1_loss(v_t, u_t)
        return loss_fm
    
    def sample_ode(self, x_0, total_load, n_steps=20, use_soft_projection=False):
        """Sample from learned flow"""
        x_t = x_0
        dt = 1.0 / n_steps
        
        for step in range(n_steps):
            t = torch.full((x_t.shape[0],), step * dt, device=x_t.device)
            v_t = self.forward(x_t, t, total_load)
            x_t = x_t + dt * v_t
            
            if use_soft_projection:
                x_t = smart_project_to_feasible(x_t, total_load, self.gen_pmin, self.gen_pmax)
            else:
                x_t = hard_project_to_feasible(x_t, total_load, self.gen_pmin, self.gen_pmax)
        
        return x_t

# ==============================================================================
# GNN
# ==============================================================================

class FastPhysicsGNN(nn.Module):
    def __init__(self, n_buses: int, n_gens: int, hidden_dim: int = 128):
        super().__init__()
        self.n_gens = n_gens

        self.load_embed = nn.Linear(1, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.dispatch_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, loads, edge_index, gen_bus_idx, batch_size):
        x = F.relu(self.load_embed(loads.view(-1, 1)))
        x = F.relu(self.norm1(self.conv1(x, edge_index)))
        x = F.relu(self.norm2(self.conv2(x, edge_index)))
        
        x = x.view(batch_size, -1, x.size(-1))
        gen_features = x[:, gen_bus_idx, :]
        p_gen = self.dispatch_head(gen_features).squeeze(-1)
        
        return p_gen

# ==============================================================================
# MODEL
# ==============================================================================

class PhysicsInformedCFMDispatchModel(nn.Module):
    def __init__(self, power_system: PowerSystem, hidden_dim: int = 128):
        super().__init__()
        self.ps = power_system
        
        self.gnn = FastPhysicsGNN(power_system.n_buses, power_system.n_gens, hidden_dim)
        self.cfm_refiner = OptimizedCFMRefiner(
            power_system.n_gens,
            power_system.gen_pmin,
            power_system.gen_pmax,
            hidden_dim=256
        )
        
        self.register_buffer('gen_pmin', torch.tensor(power_system.gen_pmin, dtype=torch.float32))
        self.register_buffer('gen_pmax', torch.tensor(power_system.gen_pmax, dtype=torch.float32))
    
    def forward(self, loads, edge_index, refine=True, n_refine_steps=20, use_soft_projection=None):
        batch_size = loads.size(0)
        n_buses = self.ps.n_buses
        total_load = loads.sum(dim=1, keepdim=True)
        
        edge_index_batched = edge_index.clone()
        for b in range(1, batch_size):
            offset = b * n_buses
            batch_edges = edge_index + offset
            edge_index_batched = torch.cat([edge_index_batched, batch_edges], dim=1)
        
        gen_bus_idx = torch.tensor(self.ps.gen_buses, dtype=torch.long, device=loads.device)
        p_gen_init = self.gnn(loads, edge_index_batched, gen_bus_idx, batch_size)
        p_gen_init = F.softplus(p_gen_init)
        p_gen_init = hard_project_to_feasible(p_gen_init, total_load, self.gen_pmin, self.gen_pmax)
        
        if not refine:
            return p_gen_init
        
        if use_soft_projection is None:
            use_soft_projection = self.training
        
        p_gen_refined = self.cfm_refiner.sample_ode(
            p_gen_init, total_load, 
            n_steps=n_refine_steps, 
            use_soft_projection=use_soft_projection
        )
        
        return p_gen_refined

# ==============================================================================
# PHYSICS-INFORMED LOSSES FOR PHASE 1
# ==============================================================================

class PhysicsInformedPhase1Loss:
    """
    TRULY PHYSICS-INFORMED LOSS combining:
    1. Economic dispatch (marginal cost equalization)
    2. DC power flow (Kirchhoff's laws)
    3. KKT complementarity conditions
    4. Progressive curriculum
    """
    
    def __init__(self, power_system: PowerSystem):
        self.ps = power_system
        self.gen_pmin = torch.tensor(power_system.gen_pmin, dtype=torch.float32)
        self.gen_pmax = torch.tensor(power_system.gen_pmax, dtype=torch.float32)
        self.gen_cost_c2 = torch.tensor(power_system.gen_cost_c2, dtype=torch.float32)
        self.gen_cost_c1 = torch.tensor(power_system.gen_cost_c1, dtype=torch.float32)
        self.gen_cost_c0 = torch.tensor(power_system.gen_cost_c0, dtype=torch.float32)
        self.gen_buses = power_system.gen_buses
    
    def compute_cost(self, p_gen):
        """Generation cost"""
        c2 = self.gen_cost_c2.to(p_gen.device)
        c1 = self.gen_cost_c1.to(p_gen.device)
        c0 = self.gen_cost_c0.to(p_gen.device)
        return (c2 * p_gen ** 2 + c1 * p_gen + c0).sum(dim=-1)
    
    def economic_dispatch_physics(self, p_gen):
        """
        PHYSICS: Economic dispatch optimality condition
        At optimum: ∂Cost/∂p_i = λ (constant) for all generators
        
        This is the LAGRANGE MULTIPLIER from the optimization problem!
        """
        c2 = self.gen_cost_c2.to(p_gen.device)
        c1 = self.gen_cost_c1.to(p_gen.device)
        
        # Marginal costs: ∂Cost/∂p_i = 2*c2_i*p_i + c1_i
        marginal_costs = 2.0 * c2.unsqueeze(0) * p_gen + c1.unsqueeze(0)
        
        # At optimum, all equal to lambda
        lambda_target = marginal_costs.mean(dim=1, keepdim=True)
        
        # Physics loss: deviation from equal marginal costs
        loss_physics = ((marginal_costs - lambda_target) ** 2).mean()
        
        return loss_physics
    
    def power_balance_physics(self, p_gen, loads):
        """
        PHYSICS: Kirchhoff's Current Law
        Sum of generation = Sum of load
        """
        power_balance = (p_gen.sum(dim=-1) - loads.sum(dim=-1)) ** 2
        return power_balance.mean()
    
    def kkt_complementarity(self, p_gen):
        """
        PHYSICS: KKT complementarity conditions
        - If p = pmin, then λ - μ_min = MC (μ_min > 0)
        - If p = pmax, then λ + μ_max = MC (μ_max > 0)
        - If pmin < p < pmax, then λ = MC (μ_min = μ_max = 0)
        """
        c2 = self.gen_cost_c2.to(p_gen.device)
        c1 = self.gen_cost_c1.to(p_gen.device)
        pmin = self.gen_pmin.to(p_gen.device)
        pmax = self.gen_pmax.to(p_gen.device)
        
        marginal_costs = 2.0 * c2.unsqueeze(0) * p_gen + c1.unsqueeze(0)
        lambda_avg = marginal_costs.mean(dim=1, keepdim=True)
        
        # At lower bound: MC should be ≤ λ (expensive generator at minimum)
        at_min = (p_gen - pmin.unsqueeze(0)).abs() < 1.0
        loss_min = F.relu(marginal_costs - lambda_avg)[at_min].mean() if at_min.any() else 0.0
        
        # At upper bound: MC should be ≥ λ (cheap generator at maximum)
        at_max = (pmax.unsqueeze(0) - p_gen).abs() < 1.0
        loss_max = F.relu(lambda_avg - marginal_costs)[at_max].mean() if at_max.any() else 0.0
        
        loss_kkt = loss_min + loss_max if isinstance(loss_min, torch.Tensor) or isinstance(loss_max, torch.Tensor) else 0.0
        
        return loss_kkt if isinstance(loss_kkt, torch.Tensor) else torch.tensor(0.0, device=p_gen.device)
    
    def cost_gap_to_optimal(self, p_gen, p_gen_target):
        """Cost gap supervision (better than MSE)"""
        cost_pred = self.compute_cost(p_gen)
        cost_target = self.compute_cost(p_gen_target)
        
        relative_gap = (cost_pred - cost_target) / (cost_target + 1e-6)
        loss_gap = (relative_gap ** 2).mean()
        
        return loss_gap, cost_pred.mean().item(), cost_target.mean().item()
    
    def phase1_loss(self, p_gen, loads, p_gen_target, epoch, max_epochs):
        """
        Physics-informed loss with progressive curriculum:
        
        Stage 1 (epochs 0-8): Learn from supervision + basic constraints
        Stage 2 (epochs 9-16): Add physics (economic dispatch + KKT)
        Stage 3 (epochs 17-25): Strong cost optimization + physics
        """
        # Curriculum progress
        progress = epoch / max_epochs
        
        # === STAGE 1: BASIC CONSTRAINTS (always important) ===
        loss_balance = self.power_balance_physics(p_gen, loads)
        
        violation = (F.relu(self.gen_pmin.to(p_gen.device) - p_gen) + 
                    F.relu(p_gen - self.gen_pmax.to(p_gen.device)))
        loss_limits = (violation ** 2).mean()
        
        # === STAGE 2: PHYSICS ===
        loss_economic = self.economic_dispatch_physics(p_gen)
        loss_kkt = self.kkt_complementarity(p_gen)
        
        # === STAGE 3: COST OPTIMIZATION ===
        loss_cost_gap, cost_pred, cost_target = self.cost_gap_to_optimal(p_gen, p_gen_target)
        loss_cost_direct = self.compute_cost(p_gen).mean()
        
        # === PROGRESSIVE WEIGHTS (3-stage curriculum) ===
        # Stage 1 (0-0.33): Learn feasibility + supervision
        # Stage 2 (0.33-0.67): Add physics
        # Stage 3 (0.67-1.0): Strong cost optimization
        
        w_balance = 500.0  # Always important
        w_limits = 250.0   # Always important
        
        # Physics kicks in at stage 2
        w_economic = 20.0 if progress > 0.33 else 5.0
        w_kkt = 10.0 if progress > 0.33 else 0.0
        
        # Supervision decreases, cost increases
        w_cost_gap = 30.0 * (1.0 - progress * 0.7)  # 30 → 9
        w_cost_direct = 5.0 * (1.0 + progress * 9.0)  # 5 → 50
        
        total = (w_balance * loss_balance + 
                w_limits * loss_limits +
                w_economic * loss_economic +      # PHYSICS!
                w_kkt * loss_kkt +                 # PHYSICS!
                w_cost_gap * loss_cost_gap +      # SUPERVISION
                w_cost_direct * loss_cost_direct) # COST OPT
        
        return total, {
            'balance': loss_balance.item(),
            'limits': loss_limits.item(),
            'economic': loss_economic.item(),
            'kkt': loss_kkt.item() if isinstance(loss_kkt, torch.Tensor) else 0.0,
            'cost_pred': cost_pred,
            'cost_target': cost_target,
            'cost_gap_%': ((cost_pred - cost_target) / cost_target * 100)
        }

# ==============================================================================
# PHASE 2 LOSS (CFM)
# ==============================================================================

class Phase2CFMLoss:
    def __init__(self, power_system: PowerSystem):
        self.ps = power_system
        self.gen_pmin = torch.tensor(power_system.gen_pmin, dtype=torch.float32)
        self.gen_pmax = torch.tensor(power_system.gen_pmax, dtype=torch.float32)
        self.gen_cost_c2 = torch.tensor(power_system.gen_cost_c2, dtype=torch.float32)
        self.gen_cost_c1 = torch.tensor(power_system.gen_cost_c1, dtype=torch.float32)
        self.gen_cost_c0 = torch.tensor(power_system.gen_cost_c0, dtype=torch.float32)
    
    def compute_cost(self, p_gen):
        c2 = self.gen_cost_c2.to(p_gen.device)
        c1 = self.gen_cost_c1.to(p_gen.device)
        c0 = self.gen_cost_c0.to(p_gen.device)
        return (c2 * p_gen ** 2 + c1 * p_gen + c0).sum(dim=-1)
    
    def phase2_cfm_loss(self, p_gen_init, p_gen_refined, p_gen_target, loads, epoch=0):
        """CFM training with curriculum"""
        curriculum = min(epoch / 20.0, 1.0)
        
        loss_balance = ((p_gen_refined.sum(dim=-1) - loads.sum(dim=-1)) ** 2).mean()
        
        violation = (F.relu(self.gen_pmin.to(p_gen_refined.device) - p_gen_refined) + 
                    F.relu(p_gen_refined - self.gen_pmax.to(p_gen_refined.device)))
        loss_limits = (violation ** 2).mean()
        
        cost_init = self.compute_cost(p_gen_init)
        cost_refined = self.compute_cost(p_gen_refined)
        cost_target = self.compute_cost(p_gen_target)
        
        loss_cost = cost_refined.mean()
        loss_improvement = F.relu(cost_refined - cost_init + 1.0).mean()
        loss_distance = F.mse_loss(p_gen_refined, p_gen_target)
        
        w_balance = 50.0
        w_limits = 25.0
        w_cost = 100.0 * (1.0 + curriculum * 2.0)
        w_improvement = 50.0 * curriculum
        w_distance = 30.0 * curriculum
        
        total = (w_balance * loss_balance + 
                w_limits * loss_limits + 
                w_cost * loss_cost + 
                w_improvement * loss_improvement +
                w_distance * loss_distance)
        
        return total, {
            'balance': loss_balance.item(),
            'limits': loss_limits.item(),
            'cost': cost_refined.mean().item(),
            'improvement': loss_improvement.item(),
            'gap_to_opt': ((cost_refined.mean() - cost_target.mean()) / cost_target.mean() * 100).item()
        }

# ==============================================================================
# DATA
# ==============================================================================

def create_ieee30_system_with_physics():
    """Create system with physics parameters"""
    print("📊 Creating IEEE 30-bus system with physics...")
    
    bus_loads = np.array([0, 21.7, 2.4, 7.6, 94.2, 0, 22.8, 30.0, 0, 5.8, 
                         0, 11.2, 0, 6.2, 8.2, 3.5, 9.0, 3.2, 9.5, 2.2, 
                         17.5, 0, 3.2, 8.7, 0, 3.5, 0, 0, 2.4, 10.6])
    
    gen_buses = np.array([0, 1, 4, 7, 10, 12])
    gen_pmin = np.array([50.0, 20.0, 15.0, 10.0, 10.0, 12.0])
    gen_pmax = np.array([200.0, 80.0, 50.0, 35.0, 30.0, 40.0])
    gen_cost_c2 = np.array([0.002, 0.0175, 0.0625, 0.00834, 0.025, 0.025])
    gen_cost_c1 = np.array([2.0, 1.75, 1.0, 3.25, 3.0, 3.0])
    gen_cost_c0 = np.zeros(6)
    
    edges = [[0,1],[0,2],[1,2],[1,3],[1,4],[2,3],[2,5],[3,4],[4,5],[4,6],
             [5,6],[6,7],[7,8],[8,9],[9,10],[10,11],[11,12],[12,13],[13,14],
             [14,15],[15,16],[16,17],[17,18],[18,19],[19,20],[20,21],[21,22],
             [22,23],[23,24],[24,25],[25,26],[26,27],[27,28],[28,29]]
    
    edge_index = np.array([[e[0], e[1]] for e in edges] + [[e[1], e[0]] for e in edges]).T
    
    n_buses = 30
    n_lines = len(edges)
    
    # Simplified admittance matrix
    admittance_matrix = np.zeros((n_buses, n_buses))
    for edge in edges:
        i, j = edge
        admittance_matrix[i, j] = -1.0
        admittance_matrix[j, i] = -1.0
        admittance_matrix[i, i] += 1.0
        admittance_matrix[j, j] += 1.0
    
    return PowerSystem(
        n_buses=n_buses, n_gens=len(gen_buses), n_lines=n_lines,
        bus_loads=bus_loads, gen_buses=gen_buses, gen_pmin=gen_pmin,
        gen_pmax=gen_pmax, gen_cost_c2=gen_cost_c2, gen_cost_c1=gen_cost_c1,
        gen_cost_c0=gen_cost_c0, edge_index=edge_index,
        ptdf_matrix=np.random.randn(n_lines, n_buses) * 0.1,
        line_limits=np.ones(n_lines) * 100.0,
        b_matrix=np.eye(n_buses) * 100.0,
        admittance_matrix=admittance_matrix
    )

def generate_training_data(ps: PowerSystem, n_samples: int = 5000):
    print(f"📦 Generating {n_samples} samples (0.7-1.0x)")
    
    solver = DCOPFSolver(ps)
    base_load = ps.bus_loads.copy()
    
    loads_list = []
    p_gen_list = []
    
    for _ in tqdm(range(n_samples), desc="Solving OPF"):
        load_factor = np.random.uniform(0.7, 1.0)
        loads = base_load * load_factor
        
        result = solver.solve(loads)
        if result['success']:
            loads_list.append(loads)
            p_gen_list.append(result['p_gen'])
    
    print(f"✅ Generated {len(loads_list)} samples")
    return (torch.tensor(np.array(loads_list), dtype=torch.float32),
            torch.tensor(np.array(p_gen_list), dtype=torch.float32))

# ==============================================================================
# TRAINING
# ==============================================================================

def train_physics_informed_phase1(model, ps, loads_train, p_gen_train, epochs=25, batch_size=256):
    print(f"\n{'='*80}")
    print("🔬 PHYSICS-INFORMED PHASE 1: GNN Training")
    print("   Stage 1 (0-8):   Feasibility + Supervision")
    print("   Stage 2 (9-16):  + Economic Dispatch + KKT")
    print("   Stage 3 (17-25): + Strong Cost Optimization")
    print(f"{'='*80}")
    
    optimizer = torch.optim.Adam(model.gnn.parameters(), lr=1e-3)
    scaler = GradScaler()
    loss_fn = PhysicsInformedPhase1Loss(ps)
    edge_index = torch.tensor(ps.edge_index, dtype=torch.long, device=device)
    
    dataset = TensorDataset(loads_train, p_gen_train)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                           pin_memory=True, num_workers=2)
    
    model.train()
    start_time = time.time()
    
    for epoch in range(epochs):
        total_metrics = {'balance': 0, 'limits': 0, 'economic': 0, 'kkt': 0, 
                        'cost_pred': 0, 'cost_target': 0, 'cost_gap_%': 0}
        n_batches = 0
        
        for batch_loads, batch_targets in dataloader:
            batch_loads = batch_loads.to(device, non_blocking=True)
            batch_targets = batch_targets.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            
            with autocast():
                p_gen = model(batch_loads, edge_index, refine=False)
                loss, metrics = loss_fn.phase1_loss(p_gen, batch_loads, batch_targets, epoch, epochs)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            for key in total_metrics:
                total_metrics[key] += metrics[key]
            n_batches += 1
        
        if epoch % 5 == 0:
            avg_cost = total_metrics['cost_pred'] / n_batches
            avg_cost_opt = total_metrics['cost_target'] / n_batches
            avg_gap = total_metrics['cost_gap_%'] / n_batches
            stage = "Stage 1" if epoch < 9 else ("Stage 2" if epoch < 17 else "Stage 3")
            
            print(f"Epoch {epoch:3d} [{stage}] [{time.time()-start_time:.1f}s]: "
                  f"Cost=${avg_cost:.2f} (Opt=${avg_cost_opt:.2f}, Gap={avg_gap:.2f}%) | "
                  f"Econ={total_metrics['economic']/n_batches:.4f} | "
                  f"KKT={total_metrics['kkt']/n_batches:.4f}")
    
    print(f"✅ Phase 1 complete")
    return model

def train_cfm_phase2(model, ps, loads_train, p_gen_train, epochs=40, batch_size=256):
    print(f"\n{'='*80}")
    print("🌊 PHASE 2: CFM Refinement")
    print(f"{'='*80}")
    
    for param in model.gnn.parameters():
        param.requires_grad = False
    
    optimizer = torch.optim.AdamW(model.cfm_refiner.parameters(), lr=3e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    scaler = GradScaler()
    loss_fn = Phase2CFMLoss(ps)
    edge_index = torch.tensor(ps.edge_index, dtype=torch.long, device=device)
    
    dataset = TensorDataset(loads_train, p_gen_train)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                           pin_memory=True, num_workers=2)
    
    model.train()
    start_time = time.time()
    
    for epoch in range(epochs):
        total_metrics = {'balance': 0, 'limits': 0, 'cost': 0, 'improvement': 0, 'gap_to_opt': 0}
        n_batches = 0
        
        for batch_loads, batch_targets in dataloader:
            batch_loads = batch_loads.to(device, non_blocking=True)
            batch_targets = batch_targets.to(device, non_blocking=True)
            total_load = batch_loads.sum(dim=1, keepdim=True)
            
            optimizer.zero_grad()
            
            with autocast():
                with torch.no_grad():
                    p_gen_init = model(batch_loads, edge_index, refine=False)
                
                loss_fm = model.cfm_refiner.compute_flow_matching_loss(
                    p_gen_init, batch_targets, total_load
                )
                
                p_gen_refined = model(batch_loads, edge_index, refine=True, 
                                     n_refine_steps=20, use_soft_projection=True)
                loss_physics, metrics = loss_fn.phase2_cfm_loss(
                    p_gen_init, p_gen_refined, batch_targets, batch_loads, epoch=epoch
                )
                
                fm_weight = max(10.0 * (1.0 - epoch / epochs), 1.0)
                loss = fm_weight * loss_fm + loss_physics
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.cfm_refiner.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()
            
            for key in total_metrics:
                total_metrics[key] += metrics[key]
            n_batches += 1
        
        scheduler.step()
        
        if epoch % 5 == 0:
            print(f"Epoch {epoch:3d} [{time.time()-start_time:.1f}s]: "
                  f"Cost=${total_metrics['cost']/n_batches:.2f} | "
                  f"Gap={total_metrics['gap_to_opt']/n_batches:.2f}%")
    
    for param in model.gnn.parameters():
        param.requires_grad = True
    
    print(f"✅ Phase 2 complete")
    return model

# ==============================================================================
# COMPREHENSIVE 5-SCENARIO EVALUATION
# ==============================================================================

def compute_cost(p_gen, ps):
    return np.sum(ps.gen_cost_c2 * p_gen**2 + ps.gen_cost_c1 * p_gen + ps.gen_cost_c0)

def check_feasibility(p_gen, loads, ps):
    power_balance = abs(p_gen.sum() - loads.sum())
    gen_violations = np.maximum(ps.gen_pmin - p_gen, 0) + np.maximum(p_gen - ps.gen_pmax, 0)
    
    return {
        'power_balance_error': power_balance,
        'gen_limit_violations': gen_violations.sum(),
        'is_feasible': power_balance < 0.1 and gen_violations.max() < 0.01
    }

def evaluate_comprehensive(model, ps):
    """
    Comprehensive 5-scenario evaluation with detailed statistics
    """
    print(f"\n{'='*80}")
    print("📊 COMPREHENSIVE 5-SCENARIO EVALUATION")
    print(f"{'='*80}\n")
    
    scenarios = [
        (0.70, 0.75, "Very Low Load (70-75%)"),
        (0.83, 0.88, "Low Load (83-88%)"),
        (0.95, 1.00, "Nominal Load (95-100%)"),
        (1.10, 1.15, "High Load (110-115%)"),
        (1.25, 1.30, "Very High Load (125-130%)"),
    ]
    
    solver = DCOPFSolver(ps)
    edge = torch.tensor(ps.edge_index, dtype=torch.long, device=device)
    base_load = ps.bus_loads.copy()
    
    all_results = {}
    
    for load_min, load_max, scenario_name in scenarios:
        print(f"\n{'─'*80}")
        print(f"🔍 {scenario_name}")
        print(f"{'─'*80}")
        
        results = {
            'optimal': {'cost': []},
            'gnn_only': {'cost': [], 'feasible': []},
            'cfm_refine': {'cost': [], 'feasible': []}
        }
        
        model.eval()
        n_samples = 100
        
        with torch.no_grad():
            for _ in tqdm(range(n_samples), desc=f"Testing", leave=False):
                load_factor = np.random.uniform(load_min, load_max)
                loads_np = base_load * load_factor
                loads_torch = torch.tensor(loads_np, dtype=torch.float32, device=device).unsqueeze(0)
                
                opt_result = solver.solve(loads_np)
                if not opt_result['success']:
                    continue
                
                results['optimal']['cost'].append(opt_result['cost'])
                
                # GNN
                p_gen_gnn = model(loads_torch, edge, refine=False).cpu().numpy()[0]
                results['gnn_only']['cost'].append(compute_cost(p_gen_gnn, ps))
                results['gnn_only']['feasible'].append(check_feasibility(p_gen_gnn, loads_np, ps)['is_feasible'])
                
                # CFM
                p_gen_cfm = model(loads_torch, edge, refine=True, n_refine_steps=30, 
                                 use_soft_projection=False).cpu().numpy()[0]
                results['cfm_refine']['cost'].append(compute_cost(p_gen_cfm, ps))
                results['cfm_refine']['feasible'].append(check_feasibility(p_gen_cfm, loads_np, ps)['is_feasible'])
        
        # Convert to arrays
        for method in results:
            for key in results[method]:
                results[method][key] = np.array(results[method][key])
        
        # Print statistics
        print_scenario_stats(results, scenario_name)
        all_results[scenario_name] = results
    
    # Summary table
    print(f"\n{'='*80}")
    print("📈 SUMMARY TABLE")
    print(f"{'='*80}\n")
    print_summary_table(all_results)
    
    return all_results

def print_scenario_stats(results, scenario_name):
    """Print detailed statistics for one scenario"""
    
    opt_costs = results['optimal']['cost']
    gnn_costs = results['gnn_only']['cost']
    cfm_costs = results['cfm_refine']['cost']
    
    opt_mean = np.mean(opt_costs)
    gnn_mean = np.mean(gnn_costs)
    cfm_mean = np.mean(cfm_costs)
    
    gnn_gap = (gnn_mean - opt_mean) / opt_mean * 100
    cfm_gap = (cfm_mean - opt_mean) / opt_mean * 100
    improvement = (gnn_mean - cfm_mean) / gnn_mean * 100
    
    print(f"\n💰 COST:")
    print(f"   Optimal:    ${opt_mean:7.2f} ± ${np.std(opt_costs):5.2f}")
    print(f"   GNN:        ${gnn_mean:7.2f} ± ${np.std(gnn_costs):5.2f}  (+{gnn_gap:5.2f}%)")
    print(f"   CFM Refine: ${cfm_mean:7.2f} ± ${np.std(cfm_costs):5.2f}  (+{cfm_gap:5.2f}%)")
    print(f"   {'🎯' if cfm_gap < 1.0 else '✅' if cfm_gap < 2.0 else '⚠️'} Improvement: {improvement:.2f}%")
    
    gnn_feas = results['gnn_only']['feasible'].mean() * 100
    cfm_feas = results['cfm_refine']['feasible'].mean() * 100
    print(f"\n✅ FEASIBILITY:")
    print(f"   GNN:        {gnn_feas:.1f}%")
    print(f"   CFM Refine: {cfm_feas:.1f}%")
    
    # Worst case
    gnn_worst_idx = np.argmax(gnn_costs - opt_costs)
    cfm_worst_idx = np.argmax(cfm_costs - opt_costs)
    gnn_worst_gap = (gnn_costs[gnn_worst_idx] - opt_costs[gnn_worst_idx]) / opt_costs[gnn_worst_idx] * 100
    cfm_worst_gap = (cfm_costs[cfm_worst_idx] - opt_costs[cfm_worst_idx]) / opt_costs[cfm_worst_idx] * 100
    
    print(f"\n⚠️  WORST-CASE:")
    print(f"   GNN:        +{gnn_worst_gap:.2f}%")
    print(f"   CFM Refine: +{cfm_worst_gap:.2f}%")

def print_summary_table(all_results):
    """Print summary across all scenarios"""
    
    print(f"{'Scenario':<30} {'GNN Gap%':<12} {'CFM Gap%':<12} {'Improve%':<12} {'CFM Feas%':<12}")
    print("─" * 78)
    
    for scenario_name, results in all_results.items():
        opt_mean = np.mean(results['optimal']['cost'])
        gnn_mean = np.mean(results['gnn_only']['cost'])
        cfm_mean = np.mean(results['cfm_refine']['cost'])
        
        gnn_gap = (gnn_mean - opt_mean) / opt_mean * 100
        cfm_gap = (cfm_mean - opt_mean) / opt_mean * 100
        improvement = (gnn_mean - cfm_mean) / gnn_mean * 100
        cfm_feas = results['cfm_refine']['feasible'].mean() * 100
        
        emoji = "🎯" if cfm_gap < 1.0 else "✅" if cfm_gap < 2.0 else "⚠️"
        print(f"{emoji} {scenario_name:<27} {gnn_gap:<12.2f} {cfm_gap:<12.2f} {improvement:<12.2f} {cfm_feas:<12.1f}")

# ==============================================================================
# MAIN
# ==============================================================================

def main():
    print("="*80)
    print("🚀 FINAL PHYSICS-INFORMED CFM WITH COMPREHENSIVE EVALUATION")
    print("="*80)
    
    ps = create_ieee30_system_with_physics()
    loads_train, p_gen_train = generate_training_data(ps, n_samples=20000)
    
    model = PhysicsInformedCFMDispatchModel(ps, hidden_dim=128).to(device)
    print(f"✅ Model: {sum(p.numel() for p in model.parameters()):,} parameters")
    
    # Phase 1: Physics-informed GNN (25 epochs, 3 stages)
    model = train_physics_informed_phase1(model, ps, loads_train, p_gen_train, epochs=40)
    
    # Phase 2: CFM refinement (40 epochs)
    model = train_cfm_phase2(model, ps, loads_train, p_gen_train, epochs=100)
    
    # Comprehensive 5-scenario evaluation
    all_results = evaluate_comprehensive(model, ps)
    
    print("\n" + "="*80)
    print("✅ COMPLETE!")
    print("="*80)
    
    return model, ps, all_results

if __name__ == "__main__":
    model, ps, all_results = main()

🖥️  Using device: cuda
🚀 FINAL PHYSICS-INFORMED CFM WITH COMPREHENSIVE EVALUATION
📊 Creating IEEE 30-bus system with physics...
📦 Generating 20000 samples (0.7-1.0x)


Solving OPF: 100%|██████████| 20000/20000 [01:27<00:00, 228.48it/s]


✅ Generated 20000 samples
✅ Model: 492,039 parameters

🔬 PHYSICS-INFORMED PHASE 1: GNN Training
   Stage 1 (0-8):   Feasibility + Supervision
   Stage 2 (9-16):  + Economic Dispatch + KKT
   Stage 3 (17-25): + Strong Cost Optimization
Epoch   0 [Stage 1] [2.4s]: Cost=$640.89 (Opt=$585.01, Gap=9.55%) | Econ=0.4767 | KKT=0.0000
Epoch   5 [Stage 1] [9.4s]: Cost=$641.03 (Opt=$585.14, Gap=9.55%) | Econ=0.4769 | KKT=0.0000
Epoch  10 [Stage 2] [16.4s]: Cost=$640.87 (Opt=$585.00, Gap=9.55%) | Econ=0.4767 | KKT=0.0000
Epoch  15 [Stage 2] [23.3s]: Cost=$640.76 (Opt=$584.90, Gap=9.55%) | Econ=0.4766 | KKT=0.0000
Epoch  20 [Stage 3] [30.3s]: Cost=$640.83 (Opt=$584.97, Gap=9.55%) | Econ=0.4766 | KKT=0.0000
Epoch  25 [Stage 3] [37.2s]: Cost=$640.86 (Opt=$584.99, Gap=9.55%) | Econ=0.4767 | KKT=0.0000
Epoch  30 [Stage 3] [44.2s]: Cost=$640.67 (Opt=$584.83, Gap=9.55%) | Econ=0.4765 | KKT=0.0000
Epoch  35 [Stage 3] [51.2s]: Cost=$640.82 (Opt=$584.95, Gap=9.55%) | Econ=0.4766 | KKT=0.0000
✅ Phase 1 compl


💰 COST:
   Optimal:    $ 490.24 ± $10.32
   GNN:        $ 528.80 ± $12.20  (+ 7.86%)
   CFM Refine: $ 490.39 ± $10.36  (+ 0.03%)
   🎯 Improvement: 7.26%

✅ FEASIBILITY:
   GNN:        100.0%
   CFM Refine: 100.0%

⚠️  WORST-CASE:
   GNN:        +8.25%
   CFM Refine: +0.04%

────────────────────────────────────────────────────────────────────────────────
🔍 Low Load (83-88%)
────────────────────────────────────────────────────────────────────────────────



💰 COST:
   Optimal:    $ 588.10 ± $10.68
   GNN:        $ 644.52 ± $12.64  (+ 9.60%)
   CFM Refine: $ 588.39 ± $10.68  (+ 0.05%)
   🎯 Improvement: 8.71%

✅ FEASIBILITY:
   GNN:        100.0%
   CFM Refine: 100.0%

⚠️  WORST-CASE:
   GNN:        +9.86%
   CFM Refine: +0.05%

────────────────────────────────────────────────────────────────────────────────
🔍 Nominal Load (95-100%)
────────────────────────────────────────────────────────────────────────────────



💰 COST:
   Optimal:    $ 680.75 ± $11.44
   GNN:        $ 754.10 ± $13.41  (+10.78%)
   CFM Refine: $ 681.20 ± $11.69  (+ 0.07%)
   🎯 Improvement: 9.67%

✅ FEASIBILITY:
   GNN:        100.0%
   CFM Refine: 100.0%

⚠️  WORST-CASE:
   GNN:        +10.92%
   CFM Refine: +0.16%

────────────────────────────────────────────────────────────────────────────────
🔍 High Load (110-115%)
────────────────────────────────────────────────────────────────────────────────



💰 COST:
   Optimal:    $ 823.24 ± $14.68
   GNN:        $ 901.26 ± $14.26  (+ 9.48%)
   CFM Refine: $ 844.59 ± $21.81  (+ 2.59%)
   ⚠️ Improvement: 6.29%

✅ FEASIBILITY:
   GNN:        100.0%
   CFM Refine: 100.0%

⚠️  WORST-CASE:
   GNN:        +9.89%
   CFM Refine: +3.60%

────────────────────────────────────────────────────────────────────────────────
🔍 Very High Load (125-130%)
────────────────────────────────────────────────────────────────────────────────



💰 COST:
   Optimal:    $ 982.93 ± $16.11
   GNN:        $1053.40 ± $14.92  (+ 7.17%)
   CFM Refine: $1010.88 ± $14.99  (+ 2.84%)
   ⚠️ Improvement: 4.04%

✅ FEASIBILITY:
   GNN:        100.0%
   CFM Refine: 100.0%

⚠️  WORST-CASE:
   GNN:        +7.60%
   CFM Refine: +3.15%

📈 SUMMARY TABLE

Scenario                       GNN Gap%     CFM Gap%     Improve%     CFM Feas%   
──────────────────────────────────────────────────────────────────────────────
🎯 Very Low Load (70-75%)      7.86         0.03         7.26         100.0       
🎯 Low Load (83-88%)           9.60         0.05         8.71         100.0       
🎯 Nominal Load (95-100%)      10.78        0.07         9.67         100.0       
⚠️ High Load (110-115%)        9.48         2.59         6.29         100.0       
⚠️ Very High Load (125-130%)   7.17         2.84         4.04         100.0       

✅ COMPLETE!
